In [163]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import random

In [164]:
with open('input.txt', 'r') as f:
    text = f.read()

In [165]:
print(f'length of dataset in characters: {len(text)}')

length of dataset in characters: 1115394


In [166]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
n_emb = 32
print(f'vocab size: {vocab_size}')

vocab size: 65


In [167]:
stoi = {s:i for i,s in enumerate(chars)}
itos = {i:s for i,s in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode('hii there'))
print(decode(encode('hii there')))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [168]:
data = torch.tensor(encode(text), dtype=torch.long)

In [169]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(train_data.shape, val_data.shape)

torch.Size([1003854]) torch.Size([111540])


In [ ]:
# block_size = 8
# train_data[:block_size+1]

# x = train_data[:block_size]
# y = train_data[1:block_size+1]
# for t in range(block_size):
#     context = x[:t+1]
#     target = y[t]
#     print(f'when input is {context} the target: {target}')

In [170]:
block_size = 8
batch_size = 32

In [171]:
torch.manual_seed(1337)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint((len(data) - block_size), (batch_size,))
    # print(f"ix: {ix.shape}")
    x = torch.stack([data[i: i+block_size] for i in ix])
    y = torch.stack([data[i+1: i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')

In [190]:
class LayerNorm1d:
    def __init__(self, dim, eps=1e-5):
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        self.eps = eps

    def __call__(self, x):
        x_mean = x.mean(1, keepdim=True)
        x_var = x.var(1, keepdim=True, unbiased=False)
        xhat = (x - x_mean) / torch.sqrt(x_var + self.eps)
        self.out = self.gamma * xhat + self.beta
        return self.out
    
    def parameters(self):
        return [self.gamma, self.beta]

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_emb, head_size, bias=False)
        self.query = nn.Linear(n_emb, head_size, bias=False)
        self.value = nn.Linear(n_emb, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, hs)
        q = self.query(x) # (B, T, hs)

        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) * C**-0.5                         # basically ye btata ki kis token ko kis token pe kitna dhyan dena chahiye
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))    # yha par tril ka use isliye kiya hai taki future tokens pe dhyan na de
        wei = F.softmax(wei, dim=-1)                                    #yha par softmax ka use isliye kiya hai taki attention scores ko probabilities me convert kar sake

        v = self.value(x) # (B, T, hs)
        out = wei @ v # (B, T, hs)
        return out

In [184]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_emb, n_emb)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

In [200]:
class FeedForward(nn.Module):
    def __init__(self, n_emb):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_emb, 4*n_emb),
            nn.ReLU(),
            nn.Linear(4*n_emb, n_emb),
            nn.Dropout(0.1),
        )

    def forward(self, x):
        return self.net(x)

In [181]:
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_emb)            # lookup table for token embeddings
        self.position_embedding_table = nn.Embedding(block_size, n_emb)         # lookup table for position embeddings
        self.sa_heads = MultiHeadAttention(num_heads=4, head_size=n_emb//4)     # multi-head self attention
        self.ffwd = FeedForward(n_emb)                                          # feed forward network
        self.lm_head = nn.Linear(n_emb, vocab_size)   # ye final layer ha jo hume logits dega har token ke liye ki agla token kya ho sakta hai

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B, T, C)           # emb table ma lookup karke hume token embeddings mil jayenge
        pos_emb = self.position_embedding_table(torch.arange(T))        # position embeddings bhi lookup table se mil jayenge
        x = tok_emb + pos_emb                                           # yha par token embeddings aur position embeddings ko add kar diya hai taki hume dono ki information mil jaye
        x = self.sa_heads(x)                              # yha par multi-head self attention ka use karke hume har token ke liye ek naya representation mil jayega jo ki us token ke context ko dhyan me rakhte hue banaya gaya hai
        x = self.ffwd(x)                                  # yha par feed forward network ka use karke hume har token ke liye ek naya representation mil jayega
        logits = self.lm_head(x) # (B, T, vocab_size)     # yha par final layer se hume logits mil jayenge har token ke liye ki agla token kya ho sakta hai

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

m = BigramLanguageModel()
# out, loss = m(xb, yb)
# print(out.shape)

# ix = torch.zeros((1, 1), dtype=torch.long)
# print(decode(m.generate(ix, max_new_tokens=1)[0].tolist()))

In [176]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [182]:
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    print(f'est_loss {loss.item()}')
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


est_loss 4.141804218292236
est_loss 4.149418830871582
est_loss 4.143989562988281
est_loss 4.151329517364502
est_loss 4.1525068283081055
est_loss 4.153858184814453
est_loss 4.15541934967041
est_loss 4.141820430755615
est_loss 4.1534857749938965
est_loss 4.152071952819824
est_loss 4.157991409301758
est_loss 4.153911590576172
est_loss 4.142886161804199
est_loss 4.156597137451172
est_loss 4.159195899963379
est_loss 4.153515338897705
est_loss 4.157974720001221
est_loss 4.1409149169921875
est_loss 4.160274028778076
est_loss 4.1551690101623535
est_loss 4.150847434997559
est_loss 4.142786979675293
est_loss 4.1704020500183105
est_loss 4.1481852531433105
est_loss 4.15836763381958
est_loss 4.141567230224609
est_loss 4.149202346801758
est_loss 4.146371364593506
est_loss 4.159104347229004
est_loss 4.132810115814209
est_loss 4.147875785827637
est_loss 4.144985675811768
est_loss 4.146829128265381
est_loss 4.15602970123291
est_loss 4.147126197814941
est_loss 4.144485950469971
est_loss 4.15339756011962

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(context, max_new_tokens=1000)[0].tolist()))

In [179]:
eval_iters = 200
@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            logits, loss = m(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out

print(estimate_loss())

{'train': tensor(2.1863), 'val': tensor(2.2491)}


In [44]:
B, T, C = 4, 8, 2

In [75]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)  

In [ ]:
#Version 1: naive implementation with loops

xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b,t] = torch.mean(xprev, dim=0)

In [ ]:
#Version 2: using matrix operations

wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x
print(torch.allclose(xbow, xbow2))   # technically not exactly equal, but very close!

False


In [ ]:
#version 3: using softmax

trill = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(trill == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
print(torch.allclose(xbow2, xbow3))

True


In [ ]:
key.shape, x.shape

In [100]:
# Single Head Self Attention
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)  

head_size = 16
key = nn.Linear(n_emb, head_size, bias=False)
query = nn.Linear(n_emb, head_size, bias=False)
value = nn.Linear(n_emb, head_size, bias=False)

k = key(x)
q = query(x)
v = value(x)

wei = q @ k.transpose(-2, -1)
tril = torch.tril(torch.ones(T,T))

wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ v
out.shape

torch.Size([4, 8, 16])

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [194]:
class Block(nn.Module):
    def __init__(self, n_emb, num_heads=4):
        super().__init__()
        self.sa_heads = MultiHeadAttention(num_heads=num_heads, head_size=n_emb//num_heads)
        self.ffwd = FeedForward(n_emb)
        self.ln1 = nn.LayerNorm(n_emb)
        self.ln2 = nn.LayerNorm(n_emb)

    def forward(self, x):
        x = x + self.sa_heads(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_emb)            # lookup table for token embeddings
        self.position_embedding_table = nn.Embedding(block_size, n_emb)         # lookup table for position embeddings
        self.blocks = nn.Sequential(
            Block(n_emb, num_heads=4),
            Block(n_emb, num_heads=4),
            Block(n_emb, num_heads=4),
            nn.LayerNorm(n_emb),
        )                                          # feed forward network
        self.lm_head = nn.Linear(n_emb, vocab_size)   # ye final layer ha jo hume logits dega har token ke liye ki agla token kya ho sakta hai

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B, T, C)           # emb table ma lookup karke hume token embeddings mil jayenge
        pos_emb = self.position_embedding_table(torch.arange(T))        # position embeddings bhi lookup table se mil jayenge
        x = tok_emb + pos_emb                                           # yha par token embeddings aur position embeddings ko add kar diya hai taki hume dono ki information mil jaye
        x = self.blocks(x)                             
        logits = self.lm_head(x) # (B, T, vocab_size)     # yha par final layer se hume logits mil jayenge har token ke liye ki agla token kya ho sakta hai

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

m = BigramLanguageModel()
# out, loss = m(xb, yb)
# print(out.shape)

# ix = torch.zeros((1, 1), dtype=torch.long)
# print(decode(m.generate(ix, max_new_tokens=1)[0].tolist()))

In [198]:
optimizer2 = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [199]:
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    print(f'est_loss {loss.item()}')
    optimizer2.zero_grad(set_to_none=True)
    loss.backward()
    optimizer2.step()


est_loss 1.9279489517211914
est_loss 1.998876929283142
est_loss 1.9192864894866943
est_loss 1.911649465560913
est_loss 2.0258936882019043
est_loss 1.9573191404342651
est_loss 1.9096413850784302
est_loss 2.0015134811401367
est_loss 2.0283865928649902
est_loss 1.8217265605926514
est_loss 1.870902180671692
est_loss 1.8076618909835815
est_loss 2.0525529384613037
est_loss 2.0826151371002197
est_loss 1.9716544151306152
est_loss 2.0054280757904053
est_loss 1.9198793172836304
est_loss 2.031257390975952
est_loss 1.885595440864563
est_loss 1.8191860914230347
est_loss 1.9146677255630493
est_loss 1.9759835004806519
est_loss 2.0175299644470215
est_loss 1.8683509826660156
est_loss 2.010927677154541
est_loss 1.975393533706665
est_loss 2.0575690269470215
est_loss 1.7628281116485596
est_loss 1.9927594661712646
est_loss 1.8628723621368408
est_loss 1.9137117862701416
est_loss 1.975690484046936
est_loss 2.1142995357513428
est_loss 1.9701627492904663
est_loss 1.9116290807724
est_loss 2.043286085128784
est_

In [202]:
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(context, max_new_tokens=1000)[0].tolist()))


Firthis friad will twlast ere, somen to ret,? yoursh unwing, and time man't you long. Maps thou say to filingward'd,
That other clune.

KING man to the is I which guish, with mine when! Mens. Awost the look this place 'Constence.

GLOUMEO:
Youive mus!
Slight give Rordiers livind Gloat, him dain's fair a wards
Nown diff in than thereous it the stot for?

GLOUCIO:
He we with arrifling.

QUEEN Now plavish
Wely hath'd I lace lace see, am chard rebber to tars,
Light quin'd so see,
And on and vemanstandinot we were thou me in' be loves maty mest?
Which it,
Lay.

CLIFF Weich more in the enous, my it.

KING
Yen gent I sin this fight eyes: I'll he tarth my thate it, for though, to ban I?

GLOUCIO:
I pacres stay,
Threaves fign'd gixter for by not frave a pray.
I said,
Wome of thee revoring
mays
Wefich this of wout thank coulsmard,
Whithousen to eat I haven the bod them?, To any bable?

Sin my from know lear, Fray be own one,
A bill the swearl hich is of bind desen was hex rise any do hand we to